In [12]:
import pandas as pd
from openai import OpenAI

from src.config import QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY, OPENAI_MODEL, OPENAI_API_URL,\
     DENSE_EMBEDDING_MODEL_PATH, OPENAI_MODEL_MINI, INTERIM_DATA_DIR
from src.dataset import render_table_ddls, rewrite_query_descriptions_csv
from src.script_generator import generate_query_descriptions, generate_sql_scripts_and_results
from src.vanna_connector import initialize_vanna

In [9]:
sqlite_config = {
    "params": {
        "url": str("/home/dima/Sber/cursor_projects/vanna-sql/data/processed/bank_transaction_monitoring/bank_transaction_monitoring_inline_short.sqlite.db") # 
    },
    "type": "sqlite"}

qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": "cpu"}

openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL,
                 "base_url": OPENAI_API_URL}

In [3]:
vanna_client = initialize_vanna(db_config=sqlite_config,
                                qdrant_config=qdrant_config,
                                openai_config=openai_config)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2024.50it/s]
/home/dima/Sber/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(


In [ ]:
openai_client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_URL,
)

# Adding DDL to vector store

In [10]:
table_ddls = render_table_ddls(
    database_name="bank_transaction_monitoring",
    comment_style="inline",
    comment_variant="short",
)

print(table_ddls[0])

CREATE TABLE btm_cst ( -- Клиенты банка.
    c01 INTEGER, -- Идентификатор клиента.
    c02 TEXT   , -- Имя клиента.
    c03 TEXT   , -- Адрес клиента.
    c04 TEXT   , -- Код штата клиента.
    c05 TEXT     -- Телефон клиента.
);


In [ ]:
# save as example for inference
# df = pd.DataFrame({"ddl": table_ddls})
# df.to_csv(f"{INTERIM_DATA_DIR}/bank_transaction_monitoring/table_ddls_inline_short.csv", index=False)

In [ ]:
# for table_ddl in table_ddls:
#     vanna_client.train(ddl=table_ddl)

# Generating artificial query description -> SQL queries -> creating descriptions in different format -> adding them to vectore store

In [5]:
descriptions_csv_path, schema_description_path, sqlite_db_path = generate_query_descriptions(
    client=openai_client,
    model=OPENAI_MODEL,
    comment_style="inline",
    comment_variant="short",
    database_name="bank_transaction_monitoring",
    counts_by_difficulty={"easy": 20, "medium": 20, "hard": 20},
    temperature=1.0,
)

print("Descriptions CSV:", descriptions_csv_path)
print("Schema description:", schema_description_path)
print("SQLite DB:", sqlite_db_path)

Descriptions CSV: /home/dima/Sber/cursor_projects/vanna-sql/data/interim/bank_transaction_monitoring/query_descriptions_inline_short.csv
Schema description: /home/dima/Sber/cursor_projects/vanna-sql/data/interim/bank_transaction_monitoring/schema_description_inline_short.txt
SQLite DB: /home/dima/Sber/cursor_projects/vanna-sql/data/processed/bank_transaction_monitoring/bank_transaction_monitoring_inline_short.sqlite.db


In [6]:
generation_summary = generate_sql_scripts_and_results(
    client=openai_client,
    model=OPENAI_MODEL,
    sqlite_db_path=sqlite_db_path,
    interim_dir=descriptions_csv_path.parent,
    descriptions_csv_path=descriptions_csv_path,
    schema_description_path=schema_description_path,
    temperature=0.2,
)

generation_summary

{'total': 60,
 'generated': 60,
 'saved': 45,
 'failed': 2,
 'empty': 13,
 'invalid_sql': 0}

In [7]:
descriptions_csv_path = "/home/dima/Sber/cursor_projects/vanna-sql/data/interim/bank_transaction_monitoring/query_descriptions_inline_short.csv"
schema_description_path = "/home/dima/Sber/cursor_projects/vanna-sql/data/interim/bank_transaction_monitoring/schema_description_inline_short.txt"

rewritten_descriptions_csv_path = rewrite_query_descriptions_csv(
    descriptions_csv_path=descriptions_csv_path,
    client=openai_client,
    model=OPENAI_MODEL,
    schema_description_path=schema_description_path,
    rewrite_styles=("short", "business", "technical"),
    source_column="query",
    temperature=0.9,
)

print("Rewritten descriptions CSV:", rewritten_descriptions_csv_path)

Rewritten descriptions CSV: /home/dima/Sber/cursor_projects/vanna-sql/data/interim/bank_transaction_monitoring/query_descriptions_inline_short_rewritten.csv


In [10]:
sql = vanna_client.generate_sql("Верни мне все уникальные имена клиентов?")
print("Generated SQL:\n", sql)
vanna_client.run_sql(sql)

SQL Prompt: [{'role': 'system', 'content': "You are a SQLite expert. Please help to generate a SQL query to answer the question. Your response should ONLY be based on the given context and follow the response guidelines and format instructions. \n===Tables \nCREATE TABLE btm_cex ( -- Экспортная копия клиентов.\n    x01 TEXT   , -- Идентификатор клиента в выгрузке.\n    x02 TEXT   , -- Имя клиента в выгрузке.\n    x03 TEXT   , -- Адрес клиента в выгрузке.\n    x04 TEXT   , -- Код штата в выгрузке.\n    x05 TEXT     -- Телефон в выгрузке.\n);\n\nCREATE TABLE btm_cst ( -- Клиенты банка.\n    c01 INTEGER, -- Идентификатор клиента.\n    c02 TEXT   , -- Имя клиента.\n    c03 TEXT   , -- Адрес клиента.\n    c04 TEXT   , -- Код штата клиента.\n    c05 TEXT     -- Телефон клиента.\n);\n\nCREATE TABLE btm_rel ( -- Связи между счетами.\n    r01 INTEGER, -- Идентификатор клиента.\n    r02 TEXT   , -- Номер дочернего или связанного счета.\n    r03 TEXT   , -- Тип связанного счета.\n    r04 TEXT    

,c02
0,Oliver
1,George
2,Harry
3,Jack
4,Jacob
5,Noah
6,Charlie
7,Robin
8,Amelia
9,Sophia
